In [ ]:
# Mount dataset CSV file
from google.colab import drive

mount_path = '/content/drive'

drive.mount(mount_path)

# Imports

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import folium
from folium.plugins import HeatMap

In [ ]:
# Define dataset file path
file_path = "/content/drive/Shareddrives/CS122-Team-9/US_Accidents_March23.csv"

In [ ]:
columns_needed = [
    "ID",
    "Severity",           # Accident severity scale (1~4)
    "Start_Time",         # Date and time accident started
    "End_Time",           # Date and time accident ended
    "Start_Lat",          # Latitude (for map visualization)
    "Start_Lng",          # Longitude (for map visualization)
    "City",               # City where accident occurred
    "State",              # State where accident occurred
    "Weather_Condition",  # Weather at time of accident (Rain, Snow, Clear, etc.)
    "Temperature(F)",     # Temperature in Fahrenheit
    "Humidity(%)",        # Humidity percentage
    "Visibility(mi)",     # Visibility in miles
    "Wind_Speed(mph)",    # Wind speed in mph
    "Precipitation(in)",  # Precipitation amount in inches
    "Sunrise_Sunset",     # Day or Night
    "Junction",           # Whether accident occurred at a junction
    "Traffic_Signal",     # Whether a traffic signal was present
]

df = pd.read_csv(file_path, usecols=columns_needed)

In [ ]:
# Preview first 20 rows (no need to re-read the CSV)
df.head(20)

In [ ]:
# Shape, null counts, and data types
print("=== Shape ===")
print(f"Rows: {df.shape[0]:,}  |  Columns: {df.shape[1]}")

print("\n=== Null Counts ===")
print(df.isnull().sum())

print("\n=== Data Types ===")
print(df.dtypes)

In [ ]:
# Convert Start_Time and End_Time to datetime format
# errors='coerce' turns unparseable strings into NaT instead of raising
df["Start_Time"] = pd.to_datetime(df["Start_Time"], format='mixed', errors='coerce')
df["End_Time"]   = pd.to_datetime(df["End_Time"],   format='mixed', errors='coerce')

In [ ]:
# Extract time-based features for analysis
df["Hour"]      = df["Start_Time"].dt.hour
df["DayOfWeek"] = df["Start_Time"].dt.day_name()
df["Month"]     = df["Start_Time"].dt.month
df["Year"]      = df["Start_Time"].dt.year

In [ ]:
# Drop rows where critical columns are null
critical = ["Severity", "Start_Time", "Start_Lat", "Start_Lng",
            "State", "Weather_Condition", "Sunrise_Sunset"]
df_clean = df.dropna(subset=critical).copy()

# Fill Precipitation nulls with 0.0 — assumption: missing = no precipitation
df_clean["Precipitation(in)"] = df_clean["Precipitation(in)"].fillna(0.0)

# Convert boolean columns to int on df_clean (not on the source df)
bool_cols = ["Junction", "Traffic_Signal"]
df_clean[bool_cols] = df_clean[bool_cols].astype(int)

print(f"Before cleaning: {df.shape[0]:,} rows")
print(f"After cleaning:  {df_clean.shape[0]:,} rows")
print(f"Dropped: {df.shape[0] - df_clean.shape[0]:,} rows")

In [ ]:
# Accident heatmap by Hour and Day of Week
day_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

heatmap_data = df_clean.groupby(["DayOfWeek", "Hour"]).size().unstack()
heatmap_data = heatmap_data.reindex(day_order)

plt.figure(figsize=(16, 6))
sns.heatmap(heatmap_data, cmap="YlOrRd", fmt=",", annot=False, linewidths=0)
plt.title("Accident Frequency by Hour and Day of Week")
plt.xlabel("Hour of Day (0 = Midnight, 12 = Noon)")
plt.ylabel("Day of Week")
plt.tight_layout()
plt.show()

In [ ]:
# Monthly accident trend 2016–2023
monthly = df_clean.groupby(["Year", "Month"]).size().reset_index(name="count")
monthly["Date"] = pd.to_datetime(monthly[["Year", "Month"]].assign(day=1))

fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(monthly["Date"], monthly["count"], color="steelblue", linewidth=1.5)

# COVID annotation
covid_date = pd.Timestamp("2020-03-01")
ax.axvline(x=covid_date, color="red", linestyle="--", linewidth=1.5)
ax.text(covid_date, 120000, "COVID-19 (Mar 2020)", color="red", fontsize=10, ha="left")

ax.set_title("Monthly Accident Trend (2016–2023)")
ax.set_xlabel("Date")
ax.set_ylabel("Number of Accidents")
plt.tight_layout()
plt.show()

In [ ]:
# Average severity by year (2016–2023)
yearly_severity = df_clean.groupby("Year")["Severity"].mean().round(3)

plt.figure(figsize=(10, 5))
plt.plot(yearly_severity.index, yearly_severity.values, color="steelblue", marker="o", linewidth=2)
plt.title("Average Accident Severity by Year (2016–2023)")
plt.xlabel("Year")
plt.ylabel("Average Severity")
plt.xticks(yearly_severity.index)
plt.ylim(2.0, 2.5)
plt.tight_layout()
plt.show()

In [ ]:
# Top 15 weather conditions by accident count (≥1,000 records only)
weather = df_clean["Weather_Condition"].value_counts()
weather = weather[weather >= 1000].head(15)

plt.figure(figsize=(12, 6))
plt.barh(weather.index[::-1], weather.values[::-1], color="steelblue")
plt.title("Top 15 Weather Conditions During Accidents (≥1,000 records)")
plt.xlabel("Number of Accidents")
plt.ylabel("Weather Condition")
plt.tight_layout()
plt.show()

In [ ]:
# Severity composition by weather condition (top 10, ≥1000 records)
weather_counts = df_clean["Weather_Condition"].value_counts()
top_weather = weather_counts[weather_counts >= 1000].head(10).index

df_weather = df_clean[df_clean["Weather_Condition"].isin(top_weather)]

crosstab = pd.crosstab(
    df_weather["Weather_Condition"],
    df_weather["Severity"],
    normalize="index"
) * 100

crosstab.plot(kind="bar", stacked=True, figsize=(12, 6), colormap="RdYlGn_r")
plt.title("Severity Composition by Weather Condition (Top 10)")
plt.xlabel("Weather Condition")
plt.ylabel("Percentage (%)")
plt.legend(title="Severity", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# Accident count by severity level (1=minor, 4=severe)
severity = df_clean["Severity"].value_counts().sort_index()

plt.figure(figsize=(8, 5))
plt.bar(severity.index, severity.values, color="steelblue")
plt.title("Accident Count by Severity Level")
plt.xlabel("Severity (1 = Minor, 4 = Severe)")
plt.ylabel("Number of Accidents")
plt.xticks([1, 2, 3, 4])
plt.tight_layout()
plt.show()

In [ ]:
# Top 10 states by accident count
top_states = df_clean["State"].value_counts().head(10)

plt.figure(figsize=(10, 5))
plt.bar(top_states.index, top_states.values, color="steelblue")
plt.title("Top 10 States by Number of Accidents")
plt.xlabel("State")
plt.ylabel("Number of Accidents")
plt.tight_layout()
plt.show()

In [ ]:
# Per-capita accident rate by state (accidents per 100k population)
# 2022 US Census state population estimates
state_population = {
    'CA': 39029342, 'TX': 30029572, 'FL': 22244823, 'NY': 19677151,
    'PA': 12972008, 'IL': 12582032, 'OH': 11799448, 'GA': 10912876,
    'NC': 10698973, 'MI': 10034113, 'NJ':  9261699, 'VA':  8683619,
    'WA':  7785786, 'AZ':  7359197, 'TN':  7051339, 'MA':  6981974,
    'IN':  6833037, 'MO':  6177957, 'MD':  6164660, 'WI':  5893718,
    'CO':  5839926, 'MN':  5717184, 'SC':  5282634, 'AL':  5074296,
    'LA':  4590241, 'KY':  4512310, 'OR':  4240137, 'OK':  4019800,
    'CT':  3626205, 'UT':  3380800, 'IA':  3200517, 'NV':  3177772,
    'AR':  3045637, 'MS':  2961279, 'KS':  2937150, 'NM':  2113344,
    'NE':  1961504, 'ID':  1939033, 'WV':  1775156, 'HI':  1440196,
    'NH':  1395231, 'ME':  1385340, 'MT':  1122867, 'RI':  1093734,
    'DE':  1018396, 'SD':   909824, 'ND':   779261, 'AK':   733583,
    'VT':   647464, 'WY':   581381
}

# Build per-capita DataFrame using pd.Series (more concise than from_dict)
pop_series = pd.Series(state_population, name="population")
pop_df = pop_series.to_frame()
pop_df["accidents"] = df_clean["State"].value_counts()
pop_df["per_100k"]  = pop_df["accidents"] / pop_df["population"] * 100_000
pop_df = pop_df.dropna().sort_values("per_100k", ascending=False).head(15)

plt.figure(figsize=(12, 6))
plt.bar(pop_df.index, pop_df["per_100k"], color="steelblue")
plt.title("Top 15 States by Accidents per 100k Population")
plt.xlabel("State")
plt.ylabel("Accidents per 100,000 People")
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap of numerical columns
num_cols = ["Severity", "Temperature(F)", "Humidity(%)",
            "Visibility(mi)", "Wind_Speed(mph)", "Precipitation(in)",
            "Junction", "Traffic_Signal", "Hour", "Month"]

corr = df_clean[num_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", square=True)
plt.title("Correlation Heatmap of Accident Features")
plt.tight_layout()
plt.show()

In [ ]:
# Accidents by day vs night
day_night = df_clean["Sunrise_Sunset"].value_counts()

plt.figure(figsize=(6, 5))
plt.bar(day_night.index, day_night.values, color=["steelblue", "navy"])
plt.title("Accidents: Day vs Night")
plt.xlabel("Time of Day")
plt.ylabel("Number of Accidents")
plt.tight_layout()
plt.show()

In [ ]:
# Day vs Night summary table: count, mean severity, % Severity 3+4
day_night_summary = df_clean.groupby("Sunrise_Sunset").agg(
    count=("Severity", "count"),
    mean_severity=("Severity", "mean"),
    pct_severe=("Severity", lambda x: (x >= 3).sum() / len(x) * 100)
).round(2)

print("=== Day vs Night Summary ===")
print(day_night_summary)

In [ ]:
# Accident severity by Junction and Traffic Signal presence
junction_severity = df_clean.groupby("Junction")["Severity"].mean().round(2)
signal_severity   = df_clean.groupby("Traffic_Signal")["Severity"].mean().round(2)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].bar(["No Junction (0)", "Junction (1)"], junction_severity.values, color=["steelblue", "tomato"])
axes[0].set_title("Average Severity by Junction Presence")
axes[0].set_ylabel("Average Severity")
axes[0].set_ylim(2.0, 2.5)

axes[1].bar(["No Signal (0)", "Signal (1)"], signal_severity.values, color=["steelblue", "tomato"])
axes[1].set_title("Average Severity by Traffic Signal Presence")
axes[1].set_ylabel("Average Severity")
axes[1].set_ylim(2.0, 2.5)

plt.tight_layout()
plt.show()

In [ ]:
df_sample = df_clean[["Start_Lat", "Start_Lng"]].dropna().sample(50000, random_state=42)

m = folium.Map(
    location=[37.0902, -95.7129],
    zoom_start=4,
    tiles="CartoDB positron"
)

HeatMap(df_sample.values.tolist(), radius=3, blur=4).add_to(m)
m

In [ ]:
# Average severity by weather condition (Top 15, ≥1,000 records)
# Filter to ≥1,000 records first to avoid rare-condition bias
# (consistent with the weather count and composition charts above)
weather_counts = df_clean["Weather_Condition"].value_counts()
qualified = weather_counts[weather_counts >= 1000].index

weather_severity = (
    df_clean[df_clean["Weather_Condition"].isin(qualified)]
    .groupby("Weather_Condition")["Severity"]
    .mean()
    .sort_values(ascending=False)
    .head(15)
)

overall_mean = df_clean["Severity"].mean()

plt.figure(figsize=(12, 6))
plt.barh(weather_severity.index[::-1], weather_severity.values[::-1], color="steelblue")
plt.axvline(x=overall_mean, color="red", linestyle="--", linewidth=1.5,
            label=f"Overall Mean ({overall_mean:.2f})")
plt.legend()
plt.title("Average Accident Severity by Weather Condition (Top 15, ≥1,000 records)")
plt.xlabel("Average Severity")
plt.ylabel("Weather Condition")
plt.tight_layout()
plt.show()